# Week 103, Day 2: Final exam: part 2

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w103d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('bookings.py', 'w', encoding='utf-8') as f:
    f.write('import json\n\nROOMS = {"Oak": 8, "Elm": 4, "Ash": 12}\n\n\ndef parse_time(text):\n    """\'09:30\' -> minutes after midnight."""\n    hours, minutes = text.split(":")\n    return int(hours) * 60 + int(minutes)\n\n\ndef overlaps(a, b):\n    """Do two bookings (dicts with start and end in minutes) overlap?"""\n    return a["start"] <= b["end"] and b["start"] <= a["end"]\n\n\ndef book(bookings, room, start, end, people, notes=[]):\n    """Add a booking if the room exists, is big enough and is free; return it, or None."""\n    if people > ROOMS[room]:\n        return None\n    new = {"room": room, "start": parse_time(start), "end": parse_time(end), "people": people, "notes": notes}\n    for old in bookings:\n        if overlaps(old, new):\n            return None\n    bookings.append(new)\n    return new\n\n\ndef usage(bookings, room):\n    """The percentage of the working day (08:00 to 18:00) a room is booked, to 1 place."""\n    total = 0\n    for b in bookings:\n        if b["room"] == room:\n            total += b["end"] - b["start"]\n    return round(total / 600 * 100)\n\n\ndef load(path):\n    try:\n        f = open(path)\n        return json.load(f)\n    except:\n        return []\n')

*Interactive figure on the website: The timer keeps running if you reload or close the page..*

### Exercise: C: A library's loans

Write `Library`, the loans system for a small community library. Days are whole numbers (day 1, day 2 and so on); money is in **pence**, as integers. Raise `LoanError` (a subclass of `Exception` you define) with exactly the message given for each broken rule.

- **`add_book(isbn, title, copies=1)`** adds copies of a book; adding an ISBN that exists adds to its copies (and keeps the first title). **`join(member_id, name)`** adds a member; a member id that exists raises `"member exists"`.
- **`lend(isbn, member_id, day)`** lends a copy and returns the day it's due back (14 days later). Check, in this order: `"no such book"`, `"no such member"`, `"already borrowed"` (this member has this book), `"limit reached"` (3 loans at once), `"fines unpaid"` (the member owes more than 500 pence), `"no copies left"`. **`available(isbn)`** is the number of copies on the shelf.
- **`give_back(isbn, member_id, day)`** ends the loan (`"no such loan"` if there isn't one) and returns the fine for it: 20 pence for each day after the due day, at most 500 pence for one loan. Fines add to what the member owes; **`pay(member_id, pence)`** reduces it (never below 0) and returns what's left. **`owes(member_id)`** is the amount owed.
- **`overdue(day)`**: a list of `(member_id, isbn, days_late)` for loans past their due day on `day`, the most overdue first, then by member id, then by ISBN.
- **`save()`** returns the whole library as a JSON string; **`Library.load(text)`** (a class method) builds an equal library from it, loans and fines included.
- **`command(line)`** runs one line and returns a string: `add ISBN COPIES TITLE...`, `join ID NAME...`, `lend ISBN ID DAY` (gives `due DAY`), `return ISBN ID DAY` (gives `fine PENCE`), `pay ID PENCE` (gives `owes PENCE`) and `overdue DAY` (gives one line per loan, `ID ISBN DAYS`, joined with newlines, or `none`). A `LoanError` gives `error: MESSAGE`; an unknown command or the wrong number of words gives `error: bad command`.

In [ ]:
import json


class LoanError(Exception):
    pass


class Library:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def make():\n    lib = Library()\n    lib.add_book(\"111\", \"The Weir\")\n    lib.add_book(\"222\", \"Ridge Walks\", copies=2)\n    lib.add_book(\"333\", \"Mill Stories\")\n    lib.join(\"m1\", \"Ada\")\n    lib.join(\"m2\", \"Ben\")\n    return lib\n\ndef raises(action, message):\n    try:\n        action()\n    except LoanError as e:\n        assert str(e) == message, (str(e), message)\n    else:\n        raise AssertionError(f\"no LoanError: {message}\")\n\ndef test_lending():\n    \"books, members and lending (10 marks)\"\n    lib = make()\n    assert issubclass(LoanError, Exception)\n    assert lib.lend(\"222\", \"m1\", 1) == 15 and lib.available(\"222\") == 1 and lib.available(\"111\") == 1\n    assert lib.lend(\"222\", \"m2\", 3) == 17 and lib.available(\"222\") == 0\n    lib.add_book(\"222\", \"Another Title\", copies=1)\n    assert lib.available(\"222\") == 1\n    raises(lambda: lib.join(\"m1\", \"Someone\"), \"member exists\")\n\ndef test_rules():\n    \"the lending rules, in order (10 marks)\"\n    lib = make()\n    raises(lambda: lib.lend(\"999\", \"m9\", 1), \"no such book\")\n    raises(lambda: lib.lend(\"111\", \"m9\", 1), \"no such member\")\n    lib.lend(\"111\", \"m1\", 1)\n    raises(lambda: lib.lend(\"111\", \"m1\", 2), \"already borrowed\")\n    raises(lambda: lib.lend(\"111\", \"m2\", 2), \"no copies left\")\n    lib.add_book(\"444\", \"Lake Birds\")\n    lib.lend(\"222\", \"m1\", 1)\n    lib.lend(\"333\", \"m1\", 1)\n    raises(lambda: lib.lend(\"444\", \"m1\", 2), \"limit reached\")\n\ndef test_returns_and_fines():\n    \"returns, fines and payments (10 marks)\"\n    lib = make()\n    lib.lend(\"111\", \"m1\", 1)\n    assert lib.give_back(\"111\", \"m1\", 15) == 0 and lib.available(\"111\") == 1 and lib.owes(\"m1\") == 0\n    raises(lambda: lib.give_back(\"111\", \"m1\", 16), \"no such loan\")\n    lib.lend(\"111\", \"m1\", 20)\n    assert lib.give_back(\"111\", \"m1\", 38) == 80 and lib.owes(\"m1\") == 80\n    lib.lend(\"222\", \"m1\", 1)\n    assert lib.give_back(\"222\", \"m1\", 100) == 500 and lib.owes(\"m1\") == 580\n    raises(lambda: lib.lend(\"333\", \"m1\", 100), \"fines unpaid\")\n    lib.add_book(\"444\", \"Lake Birds\")\n    assert lib.pay(\"m1\", 100) == 480 and lib.lend(\"444\", \"m1\", 100) == 114\n    assert lib.pay(\"m1\", 1000) == 0 and lib.owes(\"m1\") == 0\n\ndef test_overdue_and_save():\n    \"the overdue report, saving and loading (10 marks)\"\n    lib = make()\n    lib.lend(\"111\", \"m2\", 1)\n    lib.lend(\"222\", \"m1\", 5)\n    lib.lend(\"333\", \"m1\", 1)\n    lib.lend(\"222\", \"m2\", 10)\n    assert lib.overdue(15) == []\n    assert lib.overdue(25) == [(\"m1\", \"333\", 10), (\"m2\", \"111\", 10), (\"m1\", \"222\", 6), (\"m2\", \"222\", 1)]\n    lib.give_back(\"333\", \"m1\", 25)\n    again = Library.load(lib.save())\n    assert isinstance(json.loads(lib.save()), (dict, list))\n    assert again.overdue(25) == lib.overdue(25) and again.owes(\"m1\") == 200 and again.available(\"222\") == 0\n    assert again.available(\"333\") == 1\n    raises(lambda: again.join(\"m2\", \"Ben\"), \"member exists\")\n    raises(lambda: again.lend(\"111\", \"m1\", 25), \"no copies left\")\n\ndef test_commands():\n    \"the command interpreter (10 marks)\"\n    lib = Library()\n    script = [(\"add 111 1 The Weir\", None), (\"add 222 2 Ridge Walks\", None), (\"join m1 Ada Lovelace\", None),\n              (\"join m2 Ben\", None), (\"lend 111 m1 1\", \"due 15\"), (\"lend 111 m2 1\", \"error: no copies left\"),\n              (\"lend 222 m2 2\", \"due 16\"), (\"overdue 20\", \"m1 111 5\\nm2 222 4\"), (\"return 111 m1 20\", \"fine 100\"),\n              (\"pay m1 30\", \"owes 70\"), (\"overdue 10\", \"none\"), (\"borrow 111 m1 3\", \"error: bad command\"),\n              (\"lend 111 m1\", \"error: bad command\"), (\"return 333 m1 4\", \"error: no such loan\")]\n    for line, expected in script:\n        got = lib.command(line)\n        if expected is not None:\n            assert got == expected, (line, got)\n    assert lib.available(\"222\") == 1 and lib.owes(\"m1\") == 70")

### Exercise: D: The module, reviewed and fixed

Write the corrected `bookings.py` with the same functions: `parse_time` unchanged; `overlaps` treating bookings that only touch as not overlapping; `book` returning `None` for an unknown room (no crash), comparing a new booking only with bookings **for the same room**, and giving each booking its own notes list; `usage` to 1 decimal place; `load` using `with`, returning `[]` only for a missing file and letting other errors (such as broken JSON) be raised. Above each fix, write a one-line comment saying what was wrong, as you would in a review.

In [ ]:
import json

ROOMS = {"Oak": 8, "Elm": 4, "Ash": 12}


def parse_time(text):
    hours, minutes = text.split(":")
    return int(hours) * 60 + int(minutes)


def overlaps(a, b):
    ___


def book(bookings, room, start, end, people, notes=None):
    ___


def usage(bookings, room):
    ___


def load(path):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_touching():\n    \"touching bookings don't clash (5 marks)\"\n    assert not overlaps({\"start\": 540, \"end\": 600}, {\"start\": 600, \"end\": 660})\n    assert overlaps({\"start\": 540, \"end\": 601}, {\"start\": 600, \"end\": 660})\n    assert overlaps({\"start\": 540, \"end\": 700}, {\"start\": 600, \"end\": 660})\n\ndef test_rooms():\n    \"other rooms and unknown rooms (5 marks)\"\n    bookings = []\n    assert book(bookings, \"Oak\", \"09:00\", \"10:00\", 6)[\"start\"] == 540\n    assert book(bookings, \"Elm\", \"09:00\", \"10:00\", 3) is not None\n    assert book(bookings, \"Oak\", \"09:30\", \"11:00\", 2) is None and book(bookings, \"Elm\", \"09:00\", \"10:00\", 5) is None\n    assert book(bookings, \"Oak\", \"10:00\", \"11:00\", 2) is not None\n    assert book(bookings, \"Cedar\", \"09:00\", \"10:00\", 2) is None and len(bookings) == 3\n\ndef test_notes():\n    \"each booking has its own notes (5 marks)\"\n    bookings = []\n    a = book(bookings, \"Oak\", \"09:00\", \"10:00\", 2)\n    b = book(bookings, \"Oak\", \"10:00\", \"11:00\", 2)\n    a[\"notes\"].append(\"projector\")\n    assert b[\"notes\"] == [] and a[\"notes\"] is not b[\"notes\"]\n    mine = [\"coffee\"]\n    assert book(bookings, \"Ash\", \"09:00\", \"10:00\", 2, mine)[\"notes\"] == [\"coffee\"]\n\ndef test_usage():\n    \"usage to one decimal place (5 marks)\"\n    bookings = []\n    book(bookings, \"Oak\", \"09:00\", \"09:25\", 2)\n    book(bookings, \"Ash\", \"09:00\", \"17:00\", 2)\n    assert usage(bookings, \"Oak\") == 4.2 and usage(bookings, \"Ash\") == 80.0 and usage(bookings, \"Elm\") == 0.0\n\ndef test_load():\n    \"load: missing file only, and closed (5 marks)\"\n    import json as _json\n    assert load(\"no-such-bookings.json\") == []\n    with open(\"ok.json\", \"w\") as f:\n        _json.dump([{\"room\": \"Oak\"}], f)\n    assert load(\"ok.json\") == [{\"room\": \"Oak\"}]\n    with open(\"broken.json\", \"w\") as f:\n        f.write(\"[{\")\n    try:\n        load(\"broken.json\")\n    except ValueError:\n        pass\n    else:\n        raise AssertionError(\"broken JSON was hidden\")\n    assert \"with open\" in source and \"except:\" not in source")

## Before you start

**The rules** are yesterday's: **2 hours**, alone, no notes, no tutor, no searching. Run and check as often as you like.

**The marks.**

| Part | What | Marks |
|---|---|---|
| C | A larger program: five checks of 10 marks | 50 |
| D | A code review: five checks of 5 marks | 25 |
| | **Part 2 total** | **75** |

Part C and Part D score by check, so a partly working program still earns marks: each check that passes is worth its marks.

**Strategy.** Read the whole of Part C before writing a line, and sketch the data you'll keep. Build it in the order of the checks: the first check needs only lending to work. Leave 35 minutes for Part D.

## Part C: a larger program

## Part D: a code review

`bookings.py` (on this page) books meeting rooms. A colleague has asked you to review it before it's merged. Read it as a reviewer first: try to find its five problems before you read on. They are: a booking that ends at 10:00 clashes with one starting at 10:00; `book` compares a new booking with every room's bookings, not just its own room's, and crashes on an unknown room; the notes default is one list shared by every call; `usage` rounds to a whole number instead of 1 decimal place; and `load` hides every error, not just a missing file, and leaves the file open.

## When you finish

Add up your marks for both parts: Part 1 out of 105 and Part 2 out of 75, so **180** in all. **108** or more is a pass and **144** or more a distinction. Write the two totals and your times in your log. Tomorrow you'll look back over everything you've built in the course and put your portfolio in order; whatever the score, you'll learn more from going through what you missed than from the number.